# GeoFire-X — UNE expérience : tête spécialisée nouveaux feux
U-Net base8/depth3, T=3, nouvelle tête 1×1 au décodeur pleine résolution.
L = L principal historique (poids 10) + 0,5 L nouveaux pixels (poids 10,
BCE + Dice groupé par batch). Aucun nouveau balayage de poids.
250 événements train / 93 603 tuiles, seed42, 4 époques / 23 404 updates.
La tête spécialisée est utilisée si feu_J=0 ; sinon la tête principale.
Un seul seuil global choisi par Dice micro global sur les 32 historiques.
Comparer aussi Dice et rappel des nouveaux feux à la référence sauvegardée.

**Aucun lancement ni transfert effectué par la préparation.** Attacher les 11
partitions compactes déjà publiées. Internet OFF. Ne pas attacher de test 2021.
Précontrôle possible sans GPU. Après autorisation seulement : activer un seul
T4 et RUN_TRAINING. Plafond 7 200 s de processus GPU, arrêt anticipé avec checkpoint
ou interruption externe de secours. Si incomplet : aucun score comparable à époque4.
Sélection32 séparée sur CPU, limite 3 600 s ; pas de nouvelle confirmation169 automatique.


In [ ]:
import base64, hashlib, io, json, os, sys, tempfile, time, subprocess, zipfile
from pathlib import Path
workspace=Path(tempfile.mkdtemp(prefix="geofirex-propagation-code-",dir="/tmp"))
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='90351bc94a731c09aa26e6dbbe4653ee315e9d2abac0153e239fe50f005f44ec'
with zipfile.ZipFile(io.BytesIO(payload)) as z:z.extractall(workspace)
sys.path.insert(0,str(workspace))
os.environ["PYTHONPATH"]=str(workspace)+os.pathsep+os.environ.get("PYTHONPATH", "")
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"]="0"  # Un seul GPU même si le runtime en expose deux.
import torch, numpy, psutil, matplotlib
print({"torch":torch.__version__,"numpy":numpy.__version__,"psutil":psutil.__version__,"matplotlib":matplotlib.__version__})
from geofirex.calibration_mounts import restore_verified
expected=['ed88f93ae8329c57f34709fdf3a59e3ec3272db8238f843d4329f2cb23e293d0', '4aa8b2818192a2cc67941f1c1bbb8a1001e144114486bb05ecaa9b504993c6fa', 'b5caf48618be26a75a480d91401f9de0d0936607936d09556f6308c1d76c5dcd', 'ba72fbbc58f398a7bcd471d5cb21aebe4d11ed68da02caa912c2e746ccadc04d', 'ada2f8b9a3f7f340efed521b15bf3fe4e3b4151fa39eab06ca2055518f58ab58', '3bcde37b262d5828f98153350d961e0ad771032fdedb04ff180ace536e4cb33a', '001a171f309ac58f4160269d46b9163b0c0c26c7a488cc545f6fa1b42e55d72a', '906b02993f756f4b6098ac8da11ecac173f59d20ebda70c8d2bd3bb37b8f6dbd', '793223fda8a5f0d06e792bcd09b2604a97252952a5323b144184a8e4f1496a4f', 'af3015768e0174071c4d197ad31681ef20fbe0cc1de2c1084254bfef5b91393f', '49e5551860a6d14c020c86dc33118a7cd2e0ca3dc9b470e6e95af6061817fa81']
maps=[p for p in Path("/kaggle/input").rglob("mount-map.json") if json.loads(p.read_text())["identity"] in expected]
corpus=restore_verified(maps,expected,Path(tempfile.mkdtemp(prefix="geofirex-propagation-mount-",dir="/tmp"))/"corpus")
index=corpus/"index"
OUT=Path("/kaggle/working")/("UNET_NEW_FIRE_HEAD_V1-"+time.strftime("%Y%m%d_%H%M%S"))
common=[sys.executable,"-m","geofirex.propagation_experiment","--corpus",str(corpus),"--index",str(index),
        "--protocol",str(workspace/"protocol.json"),"--baseline32",str(workspace/"baseline32.json"),"--workers","4"]
subprocess.run(common+["--out",str(OUT)],check=True,env=os.environ.copy(),timeout=120)


## Entraînement borné — uniquement après autorisation
Un seul nouveau modèle indépendant. Aucun checkpoint historique en initialisation.
Les checkpoints sauvegardent optimiseur, RNG, curseur et temps consommé ; reprise
explicite de ce même modèle seulement, sans extension du plafond cumulé 7 200 s.
Conserver les sorties Kaggle avant de fermer la session. Ne jamais relancer
un modèle déjà complet. Quatre époques ne sont pas garanties dans le plafond.


In [ ]:
RUN_TRAINING=False
RESUME=None  # Seulement un checkpoint de CETTE expérience interrompue.
if RUN_TRAINING:
    assert torch.__version__.split("+")[0]=="2.11.0", "Conserver l'environnement des entraînements indépendants précédents"
    assert numpy.__version__=="2.1.3"
    # Contrôle CPU avant tout entraînement : identique au vrai backbone initial historique.
    from geofirex.propagation_experiment import PropagationUNet, state_hash, EXPECTED_MAIN_INITIAL_SHA
    torch.manual_seed(42)
    probe=PropagationUNet()
    assert state_hash(probe,True)==EXPECTED_MAIN_INITIAL_SHA, "Initialisation différente : arrêter, aucun update autorisé par ce protocole"
    del probe
    command=common+["--out",str(OUT),"--mode","train","--execute-training"]
    consumed=0.
    if RESUME:
        state=torch.load(RESUME,map_location="cpu",weights_only=False)
        assert state["config"]["experiment"]=="UNET_NEW_FIRE_HEAD_V1"
        consumed=state["gpu_wall_seconds"]
        command += ["--resume",str(RESUME)]
    remaining=7200-int(consumed)
    assert remaining>30, "Budget épuisé : aucun nouveau lancement"
    # Timeout dur même si le chargement d'un batch se bloque. Dernier checkpoint conservé.
    try:subprocess.run(command,check=True,env=os.environ.copy(),timeout=remaining)
    except subprocess.TimeoutExpired:print("Plafond atteint ; conserver le dernier checkpoint. Aucun relancement automatique.")
else:print("GPU et entraînement désactivés.")


## Sélection32 sur CPU — nouveau modèle complet uniquement
Grille figée : 0,10–0,95 par 0,005 puis 0,96 / 0,97 / 0,98 / 0,99 / 0,995 / 0,999.
Même règle de seuil historique (Dice micro global, égalité précision puis seuil élevé).
Acceptation de développement : Dice global et Dice nouveaux feux au moins égaux
à U-Net poids10/0,78 ; rappel nouveaux feux strictement supérieur. Sinon résultat négatif,
pas de recherche supplémentaire automatique. AP et Brier restent des métriques de probabilités.
Les 32 servent au développement ; les 169 déjà vus ne sont plus un test aveugle.


In [ ]:
RUN_SELECTION32=False
CHECKPOINT=OUT/"checkpoint_latest.pt"
if RUN_SELECTION32:
    assert CHECKPOINT.is_file()
    subprocess.run(common+["--out",str(OUT)+"-selection32","--mode","select32","--checkpoint",str(CHECKPOINT)],
                   check=True,env=os.environ.copy(),timeout=3600)
else:print("Nouvelle évaluation désactivée. Aucun calcul historique répété.")
